In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
import pyquist as pq
import icm_plotly
from icm_plotly import RED, BLUE

Drag the sliders: the waveform is on the left, and the frequencies it
contains are on the right.

In [ ]:
# hide
# autorun
FC0, FM0, I0 = 220.0, 220.0, 2.0        # starting parameters

t_wave = np.linspace(0.0, 0.02, 1200)   # 20 ms of waveform
N = 4096                                # spectrum: ~93 ms, hann-windowed
sr = 44100
t_spec = np.arange(N) / sr
win = np.hanning(N)
freqs = np.fft.rfftfreq(N, 1 / sr)
mask = freqs <= 5000

def fm(fc, fm_, I, t):
    return np.sin(2 * np.pi * fc * t + I * np.sin(2 * np.pi * fm_ * t))

def spectrum(fc, fm_, I):
    X = np.abs(np.fft.rfft(fm(fc, fm_, I, t_spec) * win))
    return X[mask] / X.max()

def figure():
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12)
    fig.add_scatter(x=t_wave * 1000, y=fm(FC0, FM0, I0, t_wave),
                    mode="lines", line=dict(color=RED, width=1.8),
                    row=1, col=1)
    fig.add_scatter(x=freqs[mask], y=spectrum(FC0, FM0, I0),
                    mode="lines", line=dict(color=BLUE, width=1.5),
                    row=1, col=2)
    fig.update_xaxes(range=[0, 20], title_text="Time (ms)",
                     fixedrange=True, row=1, col=1)
    fig.update_yaxes(range=[-1.1, 1.1], title_text="Amplitude",
                     fixedrange=True, row=1, col=1)
    fig.update_xaxes(range=[0, 5000], title_text="Frequency (Hz)",
                     fixedrange=True, row=1, col=2)
    fig.update_yaxes(range=[0, 1.05], title_text="Magnitude",
                     fixedrange=True, row=1, col=2)
    return fig

def controls(fig):
    fc = widgets.FloatSlider(description="Carrier f_c (Hz)", min=55, max=880,
                             value=FC0, step=5)
    fmod = widgets.FloatSlider(description="Modulator f_m (Hz)", min=55,
                               max=880, value=FM0, step=5)
    idx = widgets.FloatSlider(description="Index I", min=0, max=10, value=I0,
                              step=0.1)

    # the defaults snapshot the arrays; the page's notebooks share one kernel
    def update(fc, fm_, I, t_wave=t_wave, fm=fm, spectrum=spectrum):
        with fig.batch_update():
            fig.data[0].y = fm(fc, fm_, I, t_wave)
            fig.data[1].y = spectrum(fc, fm_, I)

    widgets.interactive_output(update, {"fc": fc, "fm_": fmod, "I": idx})
    return widgets.VBox([fc, fmod, idx])

icm_plotly.show(figure, controls)

**Now hear it.** Edit the numbers in the cell below and re-run it: this
one decays like a bell.

In [ ]:
f_c = 220.0   # carrier (Hz)
f_m = 308.0   # modulator (Hz), try an exact multiple of f_c
I = 6.0       # index, the strength of the modulation

sr = 44100
tt = np.arange(int(2.0 * sr)) / sr
env = np.exp(-3 * tt)                     # a bell's decay
x = env * np.sin(2 * np.pi * f_c * tt + I * env * np.sin(2 * np.pi * f_m * tt))
pq.play(pq.Audio((0.5 * x).astype(np.float32), sr))